In [ ]:
pip install panel

### Documentación Detallada: Instalación de Librerías

#### Celda `0ffb2de3`: `pip install panel`

Esta celda ejecuta un comando de instalación en el entorno de Colab.

*   **`pip install panel`**: Utiliza `pip`, el gestor de paquetes de Python, para instalar la librería `panel`. `Panel` es una librería de Python de código abierto que permite crear dashboards y aplicaciones web interactivas directamente desde cuadernos Jupyter o scripts de Python. Es fundamental para construir el dashboard dinámico que se te pidió.

In [ ]:
# This cell is now empty as its content has been moved to the dashboard creation cell for self-containment.

In [ ]:
pip install jupyter_bokeh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.0 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


#### Celda `11aabd2b`: `pip install jupyter_bokeh`

Esta celda instala una dependencia adicional necesaria para la interactividad de Panel en entornos Jupyter como Google Colab.

*   **`pip install jupyter_bokeh`**: Instala la librería `jupyter_bokeh`. Aunque `Panel` es muy potente por sí solo, para que sus visualizaciones interactivas (especialmente aquellas que usan Bokeh, una librería de visualización) funcionen correctamente dentro de un cuaderno Jupyter en Colab, a menudo se requiere `jupyter_bokeh`. Esto asegura que los widgets y gráficos de tu dashboard se muestren y respondan correctamente.

### Interactive Dashboard with Panel

Now that the data is loaded and `Panel` is ready, let's create some interactive visualizations. We'll start by converting the column names to a more Python-friendly format, then create some basic plots and widgets.

In [ ]:
import pandas as pd
import panel as pn
pn.extension('tabulator') # Enable Tabulator for interactive tables in Panel
import altair as alt

# Habilitar VegaFusion para manejar grandes datasets
alt.data_transformers.enable("vegafusion")

# Load the dataset
file_path = '/content/drive/MyDrive/visualizacionDatos/universities-bolivia-dataset.csv'
df = pd.read_csv(file_path, encoding='utf-8') # Using utf-8 encoding as it's common for text data

# Clean column names for easier access
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('á', 'a').str.replace('é', 'e').str.replace('í', 'i').str.replace('ó', 'o').str.replace('ú', 'u')

# Group data by 'ciudad' and 'año' to get total matriculated students
df_grouped = df.groupby(['ciudad', 'año']).agg(
    total_matriculados_hombres=('matriculados_hombres', 'sum'),
    total_matriculados_mujeres=('matriculados_mujeres', 'sum'),
    total_nuevos_hombres=('nuevos_hombres', 'sum'),
    total_nuevos_mujeres=('nuevos_mujeres', 'sum'),
    total_titulados_hombres=('titulados_hombres', 'sum'),
    total_titulados_mujeres=('titulados_mujeres', 'sum')
).reset_index()

df_grouped['total_matriculados'] = df_grouped['total_matriculados_hombres'] + df_grouped['total_matriculados_mujeres']
df_grouped['total_nuevos'] = df_grouped['total_nuevos_hombres'] + df_grouped['total_nuevos_mujeres']
df_grouped['total_titulados'] = df_grouped['total_titulados_hombres'] + df_grouped['total_titulados_mujeres']


# Create widgets
year_slider = pn.widgets.IntSlider(name='Año', start=df['año'].min(), end=df['año'].max(), step=1, value=df['año'].min())
city_select = pn.widgets.MultiChoice(name='Ciudad', options=list(df['ciudad'].unique()), value=list(df['ciudad'].unique()))
education_type_select = pn.widgets.MultiChoice(name='Tipo de Educación', options=list(df['tipo_de_educacion'].unique()), value=list(df['tipo_de_educacion'].unique()))


@pn.depends(year_slider.param.value, city_select.param.value, education_type_select.param.value)
def filtered_data(year, cities, education_types):
    filtered_df = df[(df['año'] == year) &
                     (df['ciudad'].isin(cities)) &
                     (df['tipo_de_educacion'].isin(education_types))]
    return filtered_df

@pn.depends(year_slider.param.value, city_select.param.value, education_type_select.param.value)
def display_kpis(year, cities, education_types):
    current_filtered_df = filtered_data(year, cities, education_types)

    total_matriculados = current_filtered_df['matriculados_hombres'].sum() + current_filtered_df['matriculados_mujeres'].sum()
    total_hombres = current_filtered_df['matriculados_hombres'].sum()
    total_mujeres = current_filtered_df['matriculados_mujeres'].sum()
    total_nuevos = current_filtered_df['nuevos_hombres'].sum() + current_filtered_df['nuevos_mujeres'].sum()
    total_titulados = current_filtered_df['titulados_hombres'].sum() + current_filtered_df['titulados_mujeres'].sum()

    # Handle division by zero for percentage
    percentage_mujeres = (total_mujeres / total_matriculados * 100) if total_matriculados > 0 else 0
    percentage_titulacion = (total_titulados / total_matriculados * 100) if total_matriculados > 0 else 0 # New KPI

    return pn.Column(
        pn.Row(
            pn.indicators.Number(name='Total Matriculados', value=total_matriculados, format='{value:,}', font_size='18pt', width=200),
            pn.indicators.Number(name='Hombres Matriculados', value=total_hombres, format='{value:,}', font_size='18pt', width=200),
            pn.indicators.Number(name='Mujeres Matriculadas', value=total_mujeres, format='{value:,}', font_size='18pt', width=200)
        ),
        pn.Row(
            pn.indicators.Number(name='% Mujeres Matriculadas', value=percentage_mujeres, format='{value:.2f}%', font_size='18pt', width=200),
            pn.indicators.Number(name='Total Nuevos', value=total_nuevos, format='{value:,}', font_size='18pt', width=200),
            pn.indicators.Number(name='Total Titulados', value=total_titulados, format='{value:,}', font_size='18pt', width=200)
        ),
        pn.Row(
            pn.indicators.Number(name='% Efectivo de Titulación', value=percentage_titulacion, format='{value:.2f}%', font_size='18pt', width=200) # New KPI display
        )
    )


@pn.depends(year_slider.param.value, city_select.param.value, education_type_select.param.value)
def plot_matriculados_por_area(year, cities, education_types):
    filtered_df = filtered_data(year, cities, education_types)
    area_counts = filtered_df.groupby('area')[['matriculados_hombres', 'matriculados_mujeres']].sum().reset_index()
    area_counts['total'] = area_counts['matriculados_hombres'] + area_counts['matriculados_mujeres']
    area_counts = area_counts.sort_values('total', ascending=False)

    if area_counts.empty:
        return pn.pane.Markdown("No hay datos para la selección actual.")

    # Create a bar chart using Altair
    chart = alt.Chart(area_counts).mark_bar().encode(
        x=alt.X('total', title='Total Matriculados'),
        y=alt.Y('area', sort='-x', title='Área'),
        color='area',
        tooltip=['area', 'matriculados_hombres', 'matriculados_mujeres', 'total']
    ).properties(
        title=f'Total de Matriculados por Área en {year} (Ciudades Seleccionadas)'
    ).interactive()
    return chart.to_dict(format='vega') # Return the Altair chart object as a Vega dictionary

@pn.depends(year_slider.param.value, city_select.param.value)
def plot_matriculados_por_ciudad_anual(year, cities):
    filtered_grouped_by_city = df_grouped[(df_grouped['año'] == year) & (df_grouped['ciudad'].isin(cities))]

    if filtered_grouped_by_city.empty:
        return pn.pane.Markdown("No hay datos para la selección actual.")

    chart = alt.Chart(filtered_grouped_by_city).mark_bar().encode(
        x=alt.X('ciudad', sort=alt.EncodingSortField(field="total_matriculados", op="sum", order='descending'), title='Ciudad'),
        y=alt.Y('total_matriculados', title='Total de Matriculados'),
        tooltip=['ciudad', 'total_matriculados']
    ).properties(
        title=f'Total de Matriculados por Ciudad en {year}'
    ).interactive()
    return chart.to_dict(format='vega') # Return the Altair chart object as a Vega dictionary

@pn.depends(year_slider.param.value, city_select.param.value, education_type_select.param.value)
def plot_nuevos_vs_titulados(year, cities, education_types):
    current_filtered_df = df[(df['año'] == year) &
                             (df['ciudad'].isin(cities)) &
                             (df['tipo_de_educacion'].isin(education_types))]

    grouped_for_scatter = current_filtered_df.groupby(['ciudad', 'año']).agg(
        total_nuevos_hombres=('nuevos_hombres', 'sum'),
        total_nuevos_mujeres=('nuevos_mujeres', 'sum'),
        total_titulados_hombres=('titulados_hombres', 'sum'),
        total_titulados_mujeres=('titulados_mujeres', 'sum')
    ).reset_index()

    grouped_for_scatter['total_nuevos'] = grouped_for_scatter['total_nuevos_hombres'] + grouped_for_scatter['total_nuevos_mujeres']
    grouped_for_scatter['total_titulados'] = grouped_for_scatter['total_titulados_hombres'] + grouped_for_scatter['total_titulados_mujeres']

    if grouped_for_scatter.empty:
        return pn.pane.Markdown("No hay datos para la selección actual en este gráfico.")

    chart = alt.Chart(grouped_for_scatter).mark_point(opacity=0.7).encode(
        x=alt.X('total_nuevos', title='Total Nuevos Estudiantes'),
        y=alt.Y('total_titulados', title='Total Titulados'),
        color=alt.Color('ciudad', title='Ciudad'),
        tooltip=[
            alt.Tooltip('ciudad', title='Ciudad'),
            alt.Tooltip('año', title='Año'),
            alt.Tooltip('total_nuevos', title='Nuevos'),
            alt.Tooltip('total_titulados', title='Titulados')
        ]
    ).properties(
        title='Relación entre Nuevos Estudiantes y Titulados por Ciudad y Año'
    ).interactive()
    return chart.to_dict(format='vega')

@pn.depends(year_slider.param.value, city_select.param.value, education_type_select.param.value)
def plot_education_type_bar_chart(year, cities, education_types):
    filtered_df = filtered_data(year, cities, education_types)

    if filtered_df.empty:
        return pn.pane.Markdown("No hay datos para la selección actual en este gráfico.")

    # Agrupar por tipo de educación y sumar los matriculados totales
    education_type_counts = filtered_df.groupby('tipo_de_educacion').agg(
        total_matriculados_hombres=('matriculados_hombres', 'sum'),
        total_matriculados_mujeres=('matriculados_mujeres', 'sum')
    ).reset_index()

    education_type_counts['total_matriculados'] = education_type_counts['total_matriculados_hombres'] + education_type_counts['total_matriculados_mujeres']

    # Crear el gráfico de barras
    chart_education_type = alt.Chart(education_type_counts).mark_bar().encode(
        x=alt.X('tipo_de_educacion', title='Tipo de Educación'),
        y=alt.Y('total_matriculados', title='Total de Estudiantes Matriculados'),
        color=alt.Color('tipo_de_educacion', title='Tipo de Educación'),
        tooltip=[
            alt.Tooltip('tipo_de_educacion', title='Tipo de Educación'),
            alt.Tooltip('total_matriculados', title='Total Matriculados')
        ]
    ).properties(
        title='Total de Estudiantes Matriculados por Tipo de Educación (Pública vs. Privada)'
    ).interactive()
    return chart_education_type.to_dict(format='vega')


# Layout the dashboard following a Z-pattern
dashboard = pn.Column(
    "## Dashboard Interactivo de Universidades de Bolivia", # Top-left (start of Z)
    pn.Row( # Second part of Z, main filters and KPIs across the top
        pn.Column( # Left side: Filters
            year_slider,
            city_select,
            education_type_select
        ),
        pn.Column( # Right side: KPIs
            pn.pane.Markdown("### Indicadores Clave"),
            display_kpis
        )
    ),
    pn.Row( # Third part of Z, main plots
        pn.Column( # Left plot
            pn.pane.Markdown("### Total de Matriculados por Área"),
            pn.pane.Vega(plot_matriculados_por_area)
        ),
        pn.Column( # Right plot
            pn.pane.Markdown("### Relación Nuevos Estudiantes vs. Titulados"),
            pn.pane.Vega(plot_nuevos_vs_titulados)
        )
    ),
    pn.Row( # Bottom row, remaining plot
        pn.Column(
            pn.pane.Markdown("### Total de Matriculados por Ciudad Anual"),
            pn.pane.Vega(plot_matriculados_por_ciudad_anual),
        ),
             pn.Column(
            pn.pane.Markdown("### Total de Estudiantes Matriculados por Tipo de Educación (Pública vs. Privada)"),
            pn.pane.Vega(plot_education_type_bar_chart)
        )
    )
)

dashboard.servable()

Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] IntSlider(end=2016, label='Año', name='Año', start=2001, value=2001)
            [1] MultiChoice(label='Ciudad', name='Ciudad', options=['BENI', 'LA PAZ', ...], value=['BENI', 'LA PAZ', ...])
            [2] MultiChoice(label='Tipo de Educación', name='Tipo de Educación', options=['PUBLICA', 'PRIVADA'], value=['PUBLICA', 'PRIVADA'])
        [1] Column
            [0] Markdown(str)
            [1] ParamFunction(function, _pane=Column, defer_load=False)
    [2] Row
        [0] Column
            [0] Markdown(str)
            [1] Vega(dict, selection=Selection)
        [1] Column
            [0] Markdown(str)
            [1] Vega(dict, selection=Selection)
    [3] Row
        [0] Column
            [0] Markdown(str)
            [1] Vega(dict, selection=Selection)
        [1] Column
            [0] Markdown(str)
            [1] Vega(dict, selection=Selection)

### Ejemplos de Visualización de Conceptos de Machine Learning con Altair

Altair es excelente para crear visualizaciones interactivas y declarativas, lo que lo hace útil para entender conceptos de Machine Learning. Aquí te mostraré cómo visualizar ejemplos simples de clasificación, regresión y clustering.

#### 1. Clasificación

En clasificación, el objetivo es predecir una categoría discreta. Aquí, visualizaremos un problema de clasificación binaria simple, donde se intenta separar dos clases de puntos en un plano.

In [ ]:
import altair as alt
import pandas as pd

# Instalar vegafusion y vl-convert-python para manejar grandes datasets
!pip install "vegafusion[embed]>=1.5.0" "vl-convert-python>=1.6.0"

# Habilitar VegaFusion
alt.data_transformers.enable("vegafusion")

# Usar el DataFrame 'df' existente para el ejemplo de clasificación
# Seleccionamos las características numéricas y la variable objetivo categórica
features = ['matriculados_hombres', 'matriculados_mujeres']
target = 'tipo_de_educacion'

# Crear un subconjunto del DataFrame y eliminar filas con valores nulos en las columnas seleccionadas
data_classification = df[features + [target]].dropna()

# Convertir la columna objetivo a tipo string para que Altair la use como categoría de color
data_classification['target_label'] = data_classification[target].astype(str)

# Visualización de los puntos de datos usando Altair
chart_classification = alt.Chart(data_classification).mark_point(opacity=0.7).encode(
    x=alt.X('matriculados_hombres', title='Matriculados Hombres'),
    y=alt.Y('matriculados_mujeres', title='Matriculados Mujeres'),
    color=alt.Color('target_label', title='Tipo de Educación'), # Colorear por tipo de educación
    tooltip=[
        alt.Tooltip('matriculados_hombres', title='Hombres'),
        alt.Tooltip('matriculados_mujeres', title='Mujeres'),
        alt.Tooltip('target_label', title='Tipo de Educación')
    ]
).properties(
    title='Ejemplo de Clasificación: Tipo de Educación por Matrícula'
).interactive() # Permite hacer zoom y pan

display(chart_classification)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 16.1 MB/s eta 0:00:00


alt.Chart(...)

#### 2. Regresión

La regresión se utiliza para predecir un valor numérico continuo. Aquí, mostraremos una regresión lineal simple, donde se ajusta una línea a un conjunto de puntos de datos.

In [ ]:
import numpy as np
import altair as alt
import pandas as pd

# Usar el DataFrame 'df' existente para el ejemplo de regresión
# Seleccionamos las características numéricas y eliminamos filas con valores nulos
data_regression = df[['matriculados_hombres', 'matriculados_mujeres']].dropna()

# Asegurarse de que los datos sean numéricos
data_regression['matriculados_hombres'] = pd.to_numeric(data_regression['matriculados_hombres'])
data_regression['matriculados_mujeres'] = pd.to_numeric(data_regression['matriculados_mujeres'])

# Filtra filas donde matriculados_hombres o matriculados_mujeres es 0 para evitar divisiones por cero o logaritmos de cero, si fuera el caso
data_regression = data_regression[(data_regression['matriculados_hombres'] > 0) & (data_regression['matriculados_mujeres'] > 0)]


# Calcular la línea de regresión (aproximada para fines de visualización)
# Para una regresión más robusta, se usaría un modelo de ML real
# Usaremos np.polyfit para una línea de regresión lineal simple
if not data_regression.empty:
    poly_fit = np.polyfit(data_regression['matriculados_hombres'], data_regression['matriculados_mujeres'], 1)
    reg_line_x = np.array([data_regression['matriculados_hombres'].min(), data_regression['matriculados_hombres'].max()])
    reg_line_y = poly_fit[0] * reg_line_x + poly_fit[1]

    data_regression_line = pd.DataFrame({'x': reg_line_x, 'y': reg_line_y})

    # Visualización de los puntos de datos y la línea de regresión
    points = alt.Chart(data_regression).mark_point(opacity=0.7).encode(
        x=alt.X('matriculados_hombres', title='Matriculados Hombres'),
        y=alt.Y('matriculados_mujeres', title='Matriculados Mujeres'),
        tooltip=['matriculados_hombres', 'matriculados_mujeres']
    )

    line = alt.Chart(data_regression_line).mark_line(color='red').encode(
        x=alt.X('x', title='Matriculados Hombres'),
        y=alt.Y('y', title='Matriculados Mujeres')
    )

    chart_regression = (points + line).properties(
        title='Ejemplo de Regresión Lineal: Mujeres vs. Hombres Matriculados'
    ).interactive()
    display(chart_regression)
else:
    print("No hay datos suficientes para la regresión después de la limpieza.")

alt.LayerChart(...)

#### 3. Clustering

El clustering es una técnica de aprendizaje no supervisado para agrupar puntos de datos similares. Aquí, visualizaremos tres clusters distintos.

In [ ]:
import altair as alt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Usar el DataFrame 'df' existente para el ejemplo de clustering
# Seleccionamos las características numéricas para el clustering
features_for_clustering = ['matriculados_hombres', 'matriculados_mujeres']

# Crear un subconjunto del DataFrame y eliminar filas con valores nulos
data_clustering_raw = df[features_for_clustering].dropna().copy()

# Asegurarse de que los datos sean numéricos y positivos para evitar problemas con la escala
data_clustering_raw['matriculados_hombres'] = pd.to_numeric(data_clustering_raw['matriculados_hombres'])
data_clustering_raw['matriculados_mujeres'] = pd.to_numeric(data_clustering_raw['matriculados_mujeres'])

# Filtrar filas donde ambas columnas son cero, ya que no aportan a la diferenciación de clusters
data_clustering_raw = data_clustering_raw[(data_clustering_raw['matriculados_hombres'] > 0) | (data_clustering_raw['matriculados_mujeres'] > 0)]

if not data_clustering_raw.empty:
    # Escalar los datos para que el algoritmo de clustering funcione mejor
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data_clustering_raw[features_for_clustering])

    # Aplicar K-Means (elegir un número de clusters, por ejemplo, 3)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10) # n_init para evitar warnings
    data_clustering_raw['cluster'] = kmeans.fit_predict(scaled_features)

    # Convertir la columna de cluster a string para Altair
    data_clustering_raw['cluster_label'] = data_clustering_raw['cluster'].astype(str)

    # Visualización de los clusters
    chart_clustering = alt.Chart(data_clustering_raw).mark_point(opacity=0.7).encode(
        x=alt.X('matriculados_hombres', title='Matriculados Hombres'),
        y=alt.Y('matriculados_mujeres', title='Matriculados Mujeres'),
        color=alt.Color('cluster_label', title='Cluster'),
        tooltip=[
            alt.Tooltip('matriculados_hombres', title='Hombres'),
            alt.Tooltip('matriculados_mujeres', title='Mujeres'),
            alt.Tooltip('cluster_label', title='Cluster')
        ]
    ).properties(
        title='Ejemplo de Clustering: Universidades por Matrícula'
    ).interactive()

    display(chart_clustering)
else:
    print("No hay datos suficientes para el clustering después de la limpieza.")

alt.Chart(...)